In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [2]:
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

engine = create_engine(
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)

In [3]:
END_DAY = "20260223"

sql = """
SELECT end_day, end_time, log, status
FROM k_demon_heath_check.total_demon_report
WHERE end_day = %(end_day)s
"""

df = pd.read_sql(sql, engine, params={"end_day": END_DAY})
print(df.shape)
df.head()

(15608, 4)


,end_day,end_time,log,status
0,20260223,00:00:03,10_log,정상
1,20260223,00:00:01,11_log,정상
2,20260223,00:00:04,1_log,정상
3,20260223,00:00:03,2_log,정상
4,20260223,00:00:04,3_log,정상


In [4]:
df = df.copy()

df["end_day"] = df["end_day"].astype(str).str.strip()
df["end_time"] = df["end_time"].astype(str).str.strip()
df["log"] = df["log"].astype(str).str.strip()
df["status"] = df["status"].astype(str).str.strip()

df["ts"] = pd.to_datetime(
    df["end_day"] + " " + df["end_time"],
    format="%Y%m%d %H:%M:%S",
    errors="coerce",
)

df = df.dropna(subset=["ts"])
df = df.sort_values(["log", "ts"]).reset_index(drop=True)

df.head()

,end_day,end_time,log,status,ts
0,20260223,00:00:03,10_log,정상,2026-02-23 00:00:03
1,20260223,00:00:37,10_log,정상,2026-02-23 00:00:37
2,20260223,00:02:32,10_log,주의,2026-02-23 00:02:32
3,20260223,00:02:51,10_log,정상,2026-02-23 00:02:51
4,20260223,00:03:01,10_log,주의,2026-02-23 00:03:01


In [5]:
CASE_MAP = {
    ("정상", "정상"): "case_a_normal_and_normal",
    ("정상", "주의", "정상"): "case_b_normal_carful_normal",
    ("정상", "주의", "비정상", "정상"): "case_c_normal_carful_abnormal_normal",
    ("정상", "주의", "비정상", "비운행", "정상"): "case_d__normal_carful_abnormal_non_operation_normal",
}

def compress_consecutive(seq):
    out = []
    prev = object()
    for x in seq:
        if x != prev:
            out.append(x)
            prev = x
    return out

def extract_recover_events(group: pd.DataFrame) -> pd.DataFrame:
    """
    log 하나의 시계열에서:
    정상에서 시작해서 다시 정상으로 '복귀'하는 이벤트를 잡고,
    status 흐름(압축)이 CASE_MAP 중 하나면 recover_time 기록
    (recover_time 상한 제한 없음)
    """
    g = group.sort_values("ts").reset_index(drop=True)

    events = []
    n = len(g)
    i = 0

    while i < n - 1:
        if g.loc[i, "status"] != "정상":
            i += 1
            continue

        start_ts = g.loc[i, "ts"]

        # case_a: 바로 다음이 정상(인접 행)
        if g.loc[i + 1, "status"] == "정상":
            end_ts = g.loc[i + 1, "ts"]
            sec = (end_ts - start_ts).total_seconds()
            if sec > 0:
                events.append(
                    {
                        "log": g.loc[i, "log"],
                        "case": CASE_MAP[("정상", "정상")],
                        "start_ts": start_ts,
                        "end_ts": end_ts,
                        "recover_time": float(sec),
                    }
                )
            i += 1
            continue

        # 정상에서 벗어난 뒤, 다시 정상으로 돌아오는 첫 지점까지 추적
        j = i + 1
        while j < n and g.loc[j, "status"] != "정상":
            j += 1

        if j >= n:
            break

        end_ts = g.loc[j, "ts"]
        sec = (end_ts - start_ts).total_seconds()
        if sec <= 0:
            i += 1
            continue

        flow = compress_consecutive(g.loc[i:j, "status"].tolist())
        flow_key = tuple(flow)

        case_name = CASE_MAP.get(flow_key)
        if case_name is not None:
            events.append(
                {
                    "log": g.loc[i, "log"],
                    "case": case_name,
                    "start_ts": start_ts,
                    "end_ts": end_ts,
                    "recover_time": float(sec),
                }
            )

        i += 1

    return pd.DataFrame(events)

In [7]:
# events_df가 없거나, 이전 셀 실행이 꼬였을 때를 대비한 안전 셀

# 1) extract_recover_events 함수가 존재하는지 확인
if "extract_recover_events" not in globals():
    raise NameError("extract_recover_events 함수가 없습니다. Cell 5(함수 정의 셀)를 먼저 실행하세요.")

# 2) df가 존재하는지 확인
if "df" not in globals():
    raise NameError("df가 없습니다. Cell 3~4(데이터 로드/전처리 셀)를 먼저 실행하세요.")

events = []
for log_name, g in df.groupby("log", sort=False):
    ev = extract_recover_events(g)
    if ev is not None and not ev.empty:
        events.append(ev)

events_df = pd.concat(events, ignore_index=True) if events else pd.DataFrame(
    columns=["log", "case", "start_ts", "end_ts", "recover_time"]
)

print("events_df shape:", events_df.shape)
events_df.head(20)

events_df shape: (5653, 5)


,log,case,start_ts,end_ts,recover_time
0,10_log,case_a_normal_and_normal,2026-02-23 00:00:03,2026-02-23 00:00:37,34.0
1,10_log,case_b_normal_carful_normal,2026-02-23 00:00:37,2026-02-23 00:02:51,134.0
2,10_log,case_b_normal_carful_normal,2026-02-23 00:02:51,2026-02-23 00:03:39,48.0
3,10_log,case_b_normal_carful_normal,2026-02-23 00:03:39,2026-02-23 00:04:40,61.0
4,10_log,case_b_normal_carful_normal,2026-02-23 00:04:40,2026-02-23 00:11:46,426.0
5,10_log,case_b_normal_carful_normal,2026-02-23 00:11:46,2026-02-23 00:13:25,99.0
6,10_log,case_b_normal_carful_normal,2026-02-23 00:13:25,2026-02-23 00:18:52,327.0
7,10_log,case_b_normal_carful_normal,2026-02-23 00:18:52,2026-02-23 00:22:49,237.0
8,10_log,case_b_normal_carful_normal,2026-02-23 00:22:49,2026-02-23 00:25:39,170.0
9,10_log,case_b_normal_carful_normal,2026-02-23 00:25:39,2026-02-23 00:31:33,354.0


In [8]:
summary = (
    events_df
    .groupby(["log", "case"])["recover_time"]
    .agg(["min", "max", "count"])
    .reset_index()
    .rename(columns={"min": "min_sec", "max": "max_sec"})
)

summary.head(30)

,log,case,min_sec,max_sec,count
0,10_log,case_a_normal_and_normal,34.0,1005.0,3
1,10_log,case_b_normal_carful_normal,30.0,1497.0,163
2,11_log,case_a_normal_and_normal,34.0,34.0,1
3,11_log,case_b_normal_carful_normal,25.0,447.0,376
4,11_log,case_c_normal_carful_abnormal_normal,107.0,107.0,1
5,1_log,case_a_normal_and_normal,31.0,31.0,1
6,1_log,case_b_normal_carful_normal,22.0,662.0,277
7,1_log,case_c_normal_carful_abnormal_normal,169.0,169.0,1
8,2_log,case_a_normal_and_normal,35.0,35.0,1
9,2_log,case_b_normal_carful_normal,25.0,1194.0,210


In [11]:
# events_df가 이미 만들어져 있다는 전제
summary = (
    events_df
    .groupby(["case", "log"])["recover_time"]
    .agg(min_sec="min", max_sec="max", count="count")
    .reset_index()
)

summary.head()

,case,log,min_sec,max_sec,count
0,case_a_normal_and_normal,10_log,34.0,1005.0,3
1,case_a_normal_and_normal,11_log,34.0,34.0,1
2,case_a_normal_and_normal,1_log,31.0,31.0,1
3,case_a_normal_and_normal,2_log,35.0,35.0,1
4,case_a_normal_and_normal,3_log,34.0,34.0,1


In [12]:
case_order = [
    "case_a_normal_and_normal",
    "case_b_normal_carful_normal",
    "case_c_normal_carful_abnormal_normal",
    "case_d__normal_carful_abnormal_non_operation_normal",
]

pivot_case_log_min = (
    summary.pivot(index="case", columns="log", values="min_sec")
    .reindex(case_order)
)

pivot_case_log_max = (
    summary.pivot(index="case", columns="log", values="max_sec")
    .reindex(case_order)
)

pivot_case_log_min, pivot_case_log_max

(log                                                 10_log  11_log  1_log  \
 case                                                                        
 case_a_normal_and_normal                              34.0    34.0   31.0   
 case_b_normal_carful_normal                           30.0    25.0   22.0   
 case_c_normal_carful_abnormal_normal                   NaN   107.0  169.0   
 case_d__normal_carful_abnormal_non_operation_no...     NaN     NaN    NaN   
 
 log                                                 2_log   3_log  4_log  \
 case                                                                       
 case_a_normal_and_normal                             35.0    34.0   33.0   
 case_b_normal_carful_normal                          25.0    40.0   64.0   
 case_c_normal_carful_abnormal_normal                  NaN     NaN    NaN   
 case_d__normal_carful_abnormal_non_operation_no...    NaN  1199.0    NaN   
 
 log                                                 5_log  6_log 

In [13]:
pivot_case_log_range = (
    pivot_case_log_min.round(0).astype("Float64").astype("string")
    + " ~ "
    + pivot_case_log_max.round(0).astype("Float64").astype("string")
)

pivot_case_log_range

log,10_log,11_log,1_log,2_log,3_log,4_log,5_log,6_log,7_log,8_log,...,e2_1_log,e2_2_log,e5_log,e6_log,e7_log,f_log,gc_log,gf_log,gv_log,h_log
case,,,,,,,,,,,,,,,,,,,,,
case_a_normal_and_normal,34.0 ~ 1005.0,34.0 ~ 34.0,31.0 ~ 31.0,35.0 ~ 35.0,34.0 ~ 34.0,33.0 ~ 33.0,31.0 ~ 31.0,32.0 ~ 32.0,57.0 ~ 57.0,31.0 ~ 31.0,...,<NA>,38.0 ~ 240.0,23.0 ~ 71.0,34.0 ~ 34.0,<NA>,30.0 ~ 30.0,34.0 ~ 34.0,33.0 ~ 33.0,58.0 ~ 58.0,23.0 ~ 122.0
case_b_normal_carful_normal,30.0 ~ 1497.0,25.0 ~ 447.0,22.0 ~ 662.0,25.0 ~ 1194.0,40.0 ~ 3057.0,64.0 ~ 4129.0,35.0 ~ 2078.0,20.0 ~ 580.0,22.0 ~ 791.0,22.0 ~ 1039.0,...,<NA>,27.0 ~ 452.0,61.0 ~ 142.0,26.0 ~ 540.0,76.0 ~ 76.0,2845.0 ~ 30532.0,24.0 ~ 2416.0,39.0 ~ 1750.0,23.0 ~ 722.0,21.0 ~ 373.0
case_c_normal_carful_abnormal_normal,<NA>,107.0 ~ 107.0,169.0 ~ 169.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,105.0 ~ 105.0,61.0 ~ 192.0,106.0 ~ 302.0,<NA>,<NA>,<NA>,<NA>,<NA>,48.0 ~ 169.0
case_d__normal_carful_abnormal_non_operation_normal,<NA>,<NA>,<NA>,<NA>,1199.0 ~ 1199.0,<NA>,<NA>,<NA>,<NA>,<NA>,...,3339.0 ~ 4808.0,80.0 ~ 194.0,<NA>,<NA>,681.0 ~ 5102.0,31229.0 ~ 31229.0,<NA>,187.0 ~ 187.0,<NA>,126.0 ~ 126.0


In [14]:
from pathlib import Path
import datetime as dt

# Windows 바탕화면 경로
desktop = Path.home() / "Desktop"

# 파일명에 타임스탬프(덮어쓰기 방지)
ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

# 저장 대상(원하는 것만 남겨도 됨)
save_items = {
    "events_df": events_df,
    "summary": summary,
    "pivot_case_log_min": pivot_case_log_min,
    "pivot_case_log_max": pivot_case_log_max,
    "pivot_case_log_range": pivot_case_log_range,
}

# 1) CSV 저장
for name, _df in save_items.items():
    out_csv = desktop / f"{name}_{END_DAY}_{ts}.csv"
    _df.to_csv(out_csv, index=True, encoding="utf-8-sig")  # 한글 깨짐 방지
    print("Saved:", out_csv)

# 2) 엑셀(한 파일에 여러 시트) 저장
out_xlsx = desktop / f"recover_time_report_{END_DAY}_{ts}.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    for name, _df in save_items.items():
        # 엑셀 시트명 31자 제한 처리
        sheet = name[:31]
        _df.to_excel(writer, sheet_name=sheet, index=True)

print("Saved Excel:", out_xlsx)

Saved: C:\Users\user\Desktop\events_df_20260223_20260224_094956.csv
Saved: C:\Users\user\Desktop\summary_20260223_20260224_094956.csv
Saved: C:\Users\user\Desktop\pivot_case_log_min_20260223_20260224_094956.csv
Saved: C:\Users\user\Desktop\pivot_case_log_max_20260223_20260224_094956.csv
Saved: C:\Users\user\Desktop\pivot_case_log_range_20260223_20260224_094956.csv
Saved Excel: C:\Users\user\Desktop\recover_time_report_20260223_20260224_094956.xlsx
